# Exercise: Building a Mini LLM Serving Engine Core

Welcome to this programming exercise! In the world of LLM serving, an "engine" is the central component that orchestrates everything. It takes incoming requests, schedules them for processing on the GPU, executes the model, and formats the output. Systems like vLLM have a sophisticated `EngineCore` to manage this complex process efficiently.

In this exercise, you will implement a simplified version called `MiniEngineCore`. This will help you understand the fundamental orchestration logic at the heart of an LLM serving system.

**In this exercise you will:**
*   Implement the scheduling step of the engine's main loop.
*   Implement the model execution and output processing step.
*   Combine these pieces to create a complete `step` function that processes a batch of requests.

Let's get started!

In [ ]:
# Setup: Imports and Helper Classes
# Please do not modify the code in this cell.
# It provides the building blocks for your MiniEngineCore.

import collections
from dataclasses import dataclass, field
from enum import Enum
from typing import List, Dict

# --- Data Structures ---

class SequenceStatus(Enum):
    """Simple status for a sequence."""
    WAITING = 1
    RUNNING = 2
    FINISHED = 3

@dataclass
class SequenceGroup:
    """Represents a group of sequences from a single request."""
    request_id: str
    prompt: str
    # In a real system, this would be a list for beam search.
    # We'll keep it simple with just one sequence.
    token_ids: List[int] = field(default_factory=list)
    status: SequenceStatus = SequenceStatus.WAITING

    def __repr__(self):
        # A helper to decode token_ids for printing.
        # In our mock system, tokens are just numbers.
        decoded_text = "".join(map(str, self.token_ids))
        return f"SeqGroup(id={self.request_id}, status={self.status.name}, prompt='{self.prompt}', output='{decoded_text}')"

# --- Helper Components (Mocks) ---

class SimpleInputProcessor:
    """Turns a user prompt into a SequenceGroup."""
    def process_request(self, request_id: str, prompt: str) -> SequenceGroup:
        # In a real system, this would use a tokenizer.
        # Here, we'll just create the SequenceGroup.
        return SequenceGroup(request_id=request_id, prompt=prompt)

class SimpleScheduler:
    """Decides which requests to run next."""
    def __init__(self):
        self.waiting = collections.deque()
        self.running = {}

    def add_seq_group(self, seq_group: SequenceGroup):
        self.waiting.append(seq_group)

    def schedule(self) -> List[SequenceGroup]:
        """Schedules one waiting request to run."""
        if not self.waiting:
            return []
        
        # Move one sequence group from waiting to running.
        seq_group = self.waiting.popleft()
        seq_group.status = SequenceStatus.RUNNING
        self.running[seq_group.request_id] = seq_group
        
        print(f"Scheduler: Moved '{seq_group.request_id}' to RUNNING.")
        return [seq_group] # Return a list, like a real batch

class MockExecutor:
    """Simulates running the model on the GPU."""
    def execute_model(self, scheduled_seq_groups: List[SequenceGroup]) -> Dict[str, int]:
        """
        Simulates one forward pass.
        Returns a dictionary mapping request_id to a new generated token.
        """
        outputs = {}
        for sg in scheduled_seq_groups:
            # Generate the next token by taking the length of the current tokens.
            # e.g., if token_ids is [9, 8], next token is 2.
            next_token = len(sg.token_ids)
            outputs[sg.request_id] = next_token
            print(f"Executor: Generated token '{next_token}' for '{sg.request_id}'.")
        return outputs

class SimpleOutputProcessor:
    """Processes the model output and updates the sequence groups."""
    def process_outputs(self, scheduled_seq_groups: List[SequenceGroup], executor_outputs: Dict[str, int]):
        for sg in scheduled_seq_groups:
            if sg.request_id in executor_outputs:
                new_token = executor_outputs[sg.request_id]
                sg.token_ids.append(new_token)
                
                # Simple logic to finish a sequence
                if len(sg.token_ids) >= 5:
                    sg.status = SequenceStatus.FINISHED
                    print(f"OutputProcessor: Marked '{sg.request_id}' as FINISHED.")

print("Setup complete. All helper classes are defined.")

## The MiniEngineCore

Below is the skeleton for the `MiniEngineCore`. We have already implemented `__init__` and `add_request` for you. Your goal is to implement the methods that form the main processing loop.

The core logic of the engine happens in the `step()` method. A `step` represents one iteration of the event loop, where the engine schedules waiting requests, executes the model for running requests, and processes the results. We will build this `step` function by first implementing its component parts.

### Exercise 1: Implement `_schedule`

Your first task is to implement the `_schedule` method. This method is responsible for asking the scheduler for the next batch of sequence groups to run.

**Instructions:**
*   Call the `schedule()` method of the `self.scheduler` instance.
*   Return the result. It's that simple! This step connects the engine to the scheduler.

**Hint:** The scheduler is stored in `self.scheduler`.

In [ ]:
class MiniEngineCore:
    """
    A simplified version of vLLM's EngineCore.
    
    This class orchestrates the scheduler, executor, and processors
    to handle requests end-to-end.
    """
    def __init__(self):
        print("Initializing MiniEngineCore...")
        self.scheduler = SimpleScheduler()
        self.executor = MockExecutor()
        self.input_processor = SimpleInputProcessor()
        self.output_processor = SimpleOutputProcessor()
        self.request_counter = 0

    def add_request(self, prompt: str):
        """Adds a new request to the engine."""
        request_id = f"req-{self.request_counter}"
        self.request_counter += 1
        
        seq_group = self.input_processor.process_request(request_id, prompt)
        self.scheduler.add_seq_group(seq_group)
        print(f"Engine: Added '{request_id}' to the waiting queue.")

    def _schedule(self) -> List[SequenceGroup]:
        """
        Calls the scheduler to get the next batch of sequences to execute.
        
        Returns:
            A list of SequenceGroup objects scheduled to run in this step.
        """
        ### START CODE HERE ### (≈ 1 line)
        scheduled_seq_groups = self.scheduler.schedule()
        return scheduled_seq_groups
        ### END CODE HERE ###

    # The other methods will be implemented in the next exercises.
    def _execute_and_process(self, scheduled_seq_groups: List[SequenceGroup]) -> List[SequenceGroup]:
        pass

    def step(self) -> List[SequenceGroup]:
        pass

In [ ]:
# Test for Exercise 1: _schedule

# 1. Setup the engine
engine = MiniEngineCore()
engine.add_request(prompt="Hello, world")

# 2. Check initial state
assert len(engine.scheduler.waiting) == 1, f"Expected 1 item in waiting queue, got {len(engine.scheduler.waiting)}"
assert len(engine.scheduler.running) == 0, f"Expected 0 items in running queue, got {len(engine.scheduler.running)}"
print("Initial state is correct.")

# 3. Call the method you implemented
scheduled_batch = engine._schedule()

# 4. Check the results
assert isinstance(scheduled_batch, list), f"Expected a list, but got {type(scheduled_batch)}"
assert len(scheduled_batch) == 1, f"Expected 1 scheduled item, got {len(scheduled_batch)}"
scheduled_sg = scheduled_batch[0]
assert scheduled_sg.request_id == "req-0", f"Wrong request_id, expected 'req-0', got {scheduled_sg.request_id}"
assert scheduled_sg.status == SequenceStatus.RUNNING, f"SequenceGroup status should be RUNNING, but it is {scheduled_sg.status}"

# 5. Check the scheduler's final state
assert len(engine.scheduler.waiting) == 0, f"Expected 0 items in waiting queue after scheduling, got {len(engine.scheduler.waiting)}"
assert len(engine.scheduler.running) == 1, f"Expected 1 item in running queue after scheduling, got {len(engine.scheduler.running)}"
print("\n✅ All tests for _schedule passed!")

### Exercise 2: Implement `_execute_and_process`

Great job! Now that the engine can schedule requests, the next step is to execute them and process the results.

You will implement the `_execute_and_process` method. This method takes the batch of scheduled sequence groups and performs two actions:
1.  Calls the executor to simulate a model pass.
2.  Calls the output processor to update the sequence groups with the new tokens.

**Instructions:**
*   Call the `execute_model` method of `self.executor`, passing in the `scheduled_seq_groups`.
*   Call the `process_outputs` method of `self.output_processor`. This method needs both the `scheduled_seq_groups` and the output from the executor.
*   Return the updated `scheduled_seq_groups`.

In [ ]:
def _execute_and_process(self, scheduled_seq_groups: List[SequenceGroup]) -> List[SequenceGroup]:
    """
    Executes the model for the scheduled sequences and processes the output.
    
    Args:
        scheduled_seq_groups: The batch from the scheduler.
        
    Returns:
        The same list of SequenceGroup objects, but now updated with the
        newly generated tokens.
    """
    ### START CODE HERE ### (≈ 2-3 lines)
    if not scheduled_seq_groups:
        return []
    executor_outputs = self.executor.execute_model(scheduled_seq_groups)
    self.output_processor.process_outputs(scheduled_seq_groups, executor_outputs)
    return scheduled_seq_groups
    ### END CODE HERE ###

# We monkey-patch the method into our class for this exercise.
MiniEngineCore._execute_and_process = _execute_and_process

In [ ]:
# Test for Exercise 2: _execute_and_process

# 1. Setup the engine and a dummy scheduled batch
engine = MiniEngineCore()
# Create a SequenceGroup that looks like it has just been scheduled
test_sg = SequenceGroup(request_id="test-req", prompt="Test", status=SequenceStatus.RUNNING)
test_batch = [test_sg]

# 2. Call the method you implemented
updated_batch = engine._execute_and_process(test_batch)

# 3. Check the results
assert updated_batch is test_batch, "The method should return the same list object it received."
assert len(updated_batch) == 1, f"Expected batch size of 1, got {len(updated_batch)}"
updated_sg = updated_batch[0]
assert updated_sg.token_ids == [0], f"Expected token_ids [0], but got {updated_sg.token_ids}"

# 4. Call it again to simulate a second generation step
updated_batch_2 = engine._execute_and_process(test_batch)
assert updated_sg.token_ids == [0, 1], f"Expected token_ids [0, 1], but got {updated_sg.token_ids}"
print("\n✅ All tests for _execute_and_process passed!")

### Exercise 3: Implement `step`

You've made it to the final piece! Now you will assemble the `_schedule` and `_execute_and_process` methods into the main `step` function.

The `step` function represents a single iteration of the engine's main loop. It's the heartbeat of the serving system.

**Instructions:**
*   First, call your `_schedule()` method to get the work for this step.
*   Then, pass the result of the scheduling step to your `_execute_and_process()` method.
*   Finally, return the result from `_execute_and_process()`. This result contains the sequence groups that were processed in this step, updated with their new tokens.

In [ ]:
def step(self) -> List[SequenceGroup]:
    """
    Performs one iteration of the engine's main loop.
    
    This includes scheduling, execution, and output processing.
    
    Returns:
        A list of SequenceGroup objects that were processed in this step.
    """
    ### START CODE HERE ### (≈ 2 lines)
    scheduled_seq_groups = self._schedule()
    processed_outputs = self._execute_and_process(scheduled_seq_groups)
    return processed_outputs
    ### END CODE HERE ###
    
# Monkey-patch the final method into our class
MiniEngineCore.step = step

In [ ]:
# Integration Test: Running the Full Engine

Now let's see your `MiniEngineCore` in action! This final cell will tie everything together. We will create an engine, add a request, and then run the `step()` function multiple times to generate a sequence of tokens.

If you have implemented all the functions correctly, you should see the engine process the request from start to finish.

engine = MiniEngineCore()
engine.add_request(prompt="Once upon a time")

print("\n--- Running Engine Steps ---")
outputs = []
for i in range(5):
    print(f"\n>>> Step {i+1}")
    processed_sequences = engine.step()
    if processed_sequences:
        outputs.append(processed_sequences[0].token_ids[:]) # Make a copy

print("\n--- Final State ---")
final_sg = engine.scheduler.running["req-0"]
print(final_sg)

# --- Asserts for the integration test ---
expected_outputs = [
    [0],
    [0, 1],
    [0, 1, 2],
    [0, 1, 2, 3],
    [0, 1, 2, 3, 4]
]
assert outputs == expected_outputs, f"Generated tokens sequence is incorrect.\nGot: {outputs}\nExpected: {expected_outputs}"
assert final_sg.status == SequenceStatus.FINISHED, f"Expected final status to be FINISHED, but got {final_sg.status}"

print("\n🎉 Congratulations! You have successfully implemented the MiniEngineCore.")
print("You've built the fundamental orchestration loop of an LLM serving system!")